# Santander Product Recommendation — Full Pipeline

> **Tổng quan**: Notebook này gộp toàn bộ quy trình phân tích và xử lý dữ liệu từ 4 notebook thành một luồng hoàn chỉnh:
> - **Phần 1**: EDA trước khi làm sạch (`01_eda_before_cleaning.ipynb`)
> - **Phần 2**: Làm sạch dữ liệu & xử lý missing (`02_cleaning.ipynb`)
> - **Phần 3**: EDA sau khi làm sạch (`03_eda_after_cleaning.ipynb`)
> - **Phần 4**: Feature Engineering & định dạng Long Format (`04_feature_engineering.ipynb`)
>
> *Lưu ý*: Tất cả 4 file notebook gốc vẫn được giữ nguyên trạng độc lập trong thư mục `notebooks/`.

---


---
# ============================================================
# PHẦN 1: EDA TRƯỚC KHI LÀM SẠCH (RAW DATA EDA)
# Nguồn: `01_eda_before_cleaning.ipynb`
# Mô tả: Khám phá cấu trúc dữ liệu thô, tỷ lệ missing, tương quan và phân phối sơ bộ.
# ============================================================


## 1. Import & cấu hình
Cập nhật: bỏ `%pylab inline` (deprecated), dùng `plt.rcParams` trực tiếp.

**Fix**: logic ingest (download GCS, đọc CSV, parse basic features) đã được
tách ra `src/ingest.py` để tái sử dụng được — notebook này giờ chỉ import và
gọi, không còn tự viết logic đọc dữ liệu inline nữa.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được src/ khi chạy notebook từ notebooks/

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.ingest import download_from_gcs, peek_schema, load_raw, parse_basic_features

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

## 2. Đọc dữ liệu

### 2.1 Đọc thử 10 dòng đầu (train & test)
Peek nhanh trước khi đọc full/sample, để kiểm tra schema và cột khác nhau giữa train/test.

In [ ]:
BUCKET_NAME = "TEN-BUCKET-CUA-BAN"  # TODO: điền tên bucket thật
PROJECT = "santander-ds"
TRAIN_BLOB = "raw/train_ver2.csv"
TEST_BLOB = "raw/test_ver2.csv"

train_peek = peek_schema(BUCKET_NAME, TRAIN_BLOB, project=PROJECT, nrows=10)
test_peek = peek_schema(BUCKET_NAME, TEST_BLOB, project=PROJECT, nrows=10)

print("Train shape (10 dòng):", train_peek.shape)
print("Test shape (10 dòng):", test_peek.shape)

print("\nCột chỉ có ở train:", set(train_peek.columns) - set(test_peek.columns))
print("Cột chỉ có ở test:", set(test_peek.columns) - set(train_peek.columns))

In [ ]:
train_peek.head(10)

In [ ]:
test_peek.head(10)

### 2.2 Đọc dữ liệu train (sample để tránh crash kernel)
Tải từ GCS bucket rồi đọc, giới hạn số dòng và sample khách hàng để tránh crash kernel.
Đổi `BUCKET_NAME`/`LIMIT_ROWS`/`LIMIT_PEOPLE` cho phù hợp.

In [ ]:
LIMIT_ROWS   = 10_000_000
LIMIT_PEOPLE = 10_000
RANDOM_STATE = 42

train_csv_path = download_from_gcs(
    BUCKET_NAME, TRAIN_BLOB, "data/raw/train_ver2.csv", project=PROJECT
)
df = load_raw(train_csv_path, limit_rows=LIMIT_ROWS, limit_people=LIMIT_PEOPLE,
              random_state=RANDOM_STATE)

df.describe()

### 2.3 Parse ngày tháng & feature cơ bản
`fecha_dato` là ngày của dòng dữ liệu, `fecha_alta` là ngày khách hàng gia nhập.
Thêm cột `month` vì hành vi mua sản phẩm có thể phụ thuộc thời điểm trong năm.

In [ ]:
df = parse_basic_features(df)

df["fecha_dato"].unique()

## 3. EDA — dữ liệu thô
Trước khi ra quyết định impute/xử lý outlier cho từng cột, xem tổng quan toàn bộ dataset để quyết định có căn cứ, không phải "thấy lỗi thì sửa".

### 3.1 Tổng quan dataset

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"Số khách hàng: {df['ncodpers'].nunique()}")
print(f"Khoảng thời gian: {df['fecha_dato'].min()} -> {df['fecha_dato'].max()}")
df.dtypes.value_counts()

### 3.2 Tỷ lệ missing toàn bộ

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(10, max(4, len(missing_pct) * 0.35)))
sns.barplot(x=missing_pct.values, y=missing_pct.index, color="steelblue")
plt.xlabel("% missing")
plt.title("Tỷ lệ thiếu dữ liệu theo cột (toàn bộ dataset)")
plt.tight_layout()
plt.show()

missing_pct

### 3.3 Số dòng trùng

In [ ]:
n_dup_full = df.duplicated().sum()
n_dup_key = df.duplicated(subset=["ncodpers", "fecha_dato"]).sum()
print(f"Số dòng trùng hoàn toàn: {n_dup_full}")
print(f"Số dòng trùng theo (ncodpers, fecha_dato): {n_dup_key}")

### 3.4 Phân phối các biến numeric chính

In [ ]:
key_numeric = [c for c in ["age", "antiguedad", "renta", "indrel", "ind_actividad_cliente"] if c in df.columns]

for col in key_numeric:
    plt.figure(figsize=(10, 6))
    sns.histplot(pd.to_numeric(df[col], errors="coerce").dropna(), bins=50, color="slateblue")
    plt.title(col, fontsize=16)
    plt.xlabel(col, fontsize=13)
    plt.ylabel("Count", fontsize=13)
    plt.tight_layout()
    plt.show()

### 3.5 Tổng quan các biến categorical

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
for col in categorical_cols:
    print(f"{col}: {df[col].nunique(dropna=True)} giá trị duy nhất")

low_card_cols = [c for c in categorical_cols if df[c].nunique(dropna=True) <= 10]
for col in low_card_cols:
    plt.figure(figsize=(5, 3))
    df[col].value_counts(dropna=False).plot(kind="bar", color="coral")
    plt.title(col)
    plt.tight_layout()
    plt.show()

### 3.6 Kiểm tra missing có ngẫu nhiên không (renta theo tỉnh)

In [ ]:
df["renta_missing"] = df["renta"].isnull()
missing_rate_by_province = df.groupby("nomprov")["renta_missing"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=missing_rate_by_province.index, y=missing_rate_by_province.values, color="darkorange")
plt.xticks(rotation=90)
plt.ylabel("Tỷ lệ thiếu renta")
plt.title("Tỷ lệ missing renta theo tỉnh — missing có phải ngẫu nhiên không?")
plt.tight_layout()
plt.show()

df.drop(columns=["renta_missing"], inplace=True)

### 3.7 Tương quan giữa các biến numeric chính

In [ ]:
corr = df[key_numeric].apply(pd.to_numeric, errors="coerce").corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Tương quan giữa các biến numeric chính")
plt.tight_layout()
plt.show()

### 3.8 Số dòng dữ liệu theo tháng

In [ ]:
records_per_month = df["fecha_dato"].value_counts().sort_index()
plt.figure(figsize=(10, 4))
records_per_month.plot(kind="bar", color="teal")
plt.title("Số dòng dữ liệu theo tháng")
plt.ylabel("Số khách hàng")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.9 Phân phối số sản phẩm sở hữu / khách hàng / tháng
(Gần nhất với "target" của bài toán.)

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]
df["n_products"] = df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

plt.figure(figsize=(8, 4))
sns.histplot(df["n_products"], bins=range(0, int(df["n_products"].max()) + 2), color="mediumseagreen")
plt.title("Phân phối số sản phẩm sở hữu / khách hàng / tháng")
plt.xlabel("Số sản phẩm")
plt.tight_layout()
plt.show()

df.drop(columns=["n_products"], inplace=True)

## Checkpoint — lưu cho notebook tiếp theo
Lưu lại `df` (đã đọc + parse ngày/basic feature, CHƯA làm sạch) để
`02_cleaning.ipynb` đọc tiếp — không cần ingest lại từ GCS mỗi lần chạy notebook khác.

In [ ]:
import os
os.makedirs("data/interim", exist_ok=True)
df.to_parquet("data/interim/raw_parsed.parquet", index=False)
print(f"Đã lưu checkpoint: {df.shape}")

---
# ============================================================
# PHẦN 2: LÀM SẠCH DỮ LIỆU & XỬ LÝ MISSING (DATA CLEANING)
# Nguồn: `02_cleaning.ipynb`
# Mô tả: Làm sạch các cột age, antiguedad, renta, fecha_alta, indrel, xử lý missing và chuyển kiểu dữ liệu.
# ============================================================


# 2. Cleaning
Tiếp nối từ `01_eda_before_cleaning.ipynb` — đọc lại checkpoint `raw_parsed.parquet`
thay vì ingest lại từ GCS.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/raw_parsed.parquet")
print(f"Loaded checkpoint: {df.shape}")

## 4. Làm sạch & xử lý missing
Gộp toàn bộ các bước xử lý giá trị thiếu/outlier vào một mạch, theo đúng thứ tự phụ thuộc giữa các cột (vd: lọc `indfall` trước khi xử lý `age`, xử lý `nomprov` trước khi impute `renta` theo tỉnh).

### 4.1 Lọc khách hàng đã mất (`indfall`)
Khách hàng đã mất (`indfall == "S"`) sẽ không mua sản phẩm nào nữa trong tương
lai — giữ lại sẽ làm nhiễu dữ liệu cho bài toán dự đoán/khuyến nghị sản phẩm.
Loại các dòng này trước khi xử lý các cột khác (bao gồm cả age).

In [ ]:
print(df["indfall"].value_counts(dropna=False))

n_deceased_old = df.loc[(df["age"] > 100) & (df["indfall"] == "S")].shape[0]
print(f"Số dòng tuổi > 100 và đã mất: {n_deceased_old}")

n_before = len(df)
df = df[df["indfall"] != "S"].copy()
n_after = len(df)

print(f"Đã loại {n_before - n_after} dòng (indfall == 'S') — {((n_before - n_after) / n_before * 100):.2f}% dataset")
print(f"Shape sau khi lọc: {df.shape}")

### 4.2 Xử lý `age`
Có giá trị thiếu và outlier (tuổi quá nhỏ/quá lớn): thay
outlier bằng mean của nhóm tuổi hợp lý gần nhất, nhưng in rõ số lượng outlier để
kiểm chứng thay vì làm ngầm.

In [ ]:
n_young = int((df["age"] < 18).sum())
n_old = int((df["age"] > 100).sum())
print(f"Outliers: {n_young} tuổi < 18, {n_old} tuổi > 100")

mean_young = df.loc[(df.age >= 18) & (df.age <= 30), "age"].mean()
mean_mid = df.loc[(df.age >= 30) & (df.age <= 100), "age"].mean()

df.loc[df.age < 18, "age"] = mean_young
df.loc[df.age > 100, "age"] = mean_mid
df["age"] = df["age"].fillna(df["age"].mean())
df["age"] = df["age"].astype(int)

In [ ]:
sns.histplot(df["age"], bins=80, color="tomato")
plt.title("Age Distribution (after cleaning)")
plt.xlim(15, 100)
plt.show()

### 4.3 Xử lý `ind_nuevo` (khách hàng mới)

**Fix**: cột này được load với `dtype=str` (theo cấu hình đọc CSV ở trên), nhưng
pandas bản mới có kiểu `string` nghiêm ngặt hơn — gán thẳng số nguyên `1` vào cột
`str` sẽ ném `TypeError`. Convert sang numeric trước khi impute.

In [ ]:
df["ind_nuevo"] = pd.to_numeric(df["ind_nuevo"], errors="coerce")
n_missing = df["ind_nuevo"].isnull().sum()
print(f"Missing: {n_missing}")

if n_missing > 0:
    months_active = df.loc[df["ind_nuevo"].isnull(), :].groupby("ncodpers").size()
    print(f"Max số tháng active trong nhóm thiếu dữ liệu: {months_active.max()}")
    # Số tháng active thấp -> đúng là khách hàng mới
    df.loc[df["ind_nuevo"].isnull(), "ind_nuevo"] = 1

### 4.4 Xử lý `antiguedad` (thâm niên)
Cùng nhóm khách hàng thiếu `ind_nuevo` ở trên.

In [ ]:
df["antiguedad"] = pd.to_numeric(df["antiguedad"], errors="coerce")

# Tính thâm niên thật từ chênh lệch ngày (theo tháng, không tính theo ngày/30.44
# vì antiguedad gốc của Santander là số tháng lịch, tính kiểu này sẽ khớp)
antiguedad_calc = (
    (df["fecha_dato"].dt.year - df["fecha_alta"].dt.year) * 12
    + (df["fecha_dato"].dt.month - df["fecha_alta"].dt.month)
)

# So sánh với cột gốc để biết mức độ lệch (chỉ so trên phần antiguedad không null)
diff = (df["antiguedad"] - antiguedad_calc).dropna()
print(f"Số dòng lệch >1 tháng giữa antiguedad gốc và antiguedad tính từ ngày: {(diff.abs() > 1).sum()}")
print(diff.describe())

# Dùng bản tính từ ngày để fill missing (chính xác hơn fillna bằng min)
df.loc[df["antiguedad"].isnull(), "antiguedad"] = antiguedad_calc[df["antiguedad"].isnull()]

# Giá trị âm trong data gốc thực chất là lỗi biết trước của dataset này
# (placeholder kiểu -999999 cho khách mới) — nên thay bằng giá trị tính từ ngày
# thay vì clip cứng về 0, vì antiguedad_calc phản ánh đúng thâm niên thật
df.loc[df["antiguedad"] < 0, "antiguedad"] = antiguedad_calc[df["antiguedad"] < 0]

# Fallback cuối cùng nếu vẫn còn null (trường hợp fecha_alta cũng null)
df["antiguedad"] = df["antiguedad"].fillna(df["antiguedad"].median())

### 4.5 Xử lý `fecha_alta`
Một số dòng thiếu ngày gia nhập — gán bằng giá trị trung vị (median date).

In [ ]:
if df["fecha_alta"].isnull().any():
    mask = df["fecha_alta"].isnull() & df["antiguedad"].notnull()

    # Trừ theo tháng lịch (to_period("M") - int) rồi convert lại về timestamp,
    # vectorized nên nhanh hơn nhiều so với apply() theo từng dòng
    fecha_alta_calc = (
        df.loc[mask, "fecha_dato"].dt.to_period("M")
        - df.loc[mask, "antiguedad"].round().astype(int)
    ).dt.to_timestamp()

    df.loc[mask, "fecha_alta"] = fecha_alta_calc

    # Fallback cuối: dòng nào antiguedad cũng thiếu luôn thì mới dùng median date như cũ
    still_missing = df["fecha_alta"].isnull()
    if still_missing.any():
        median_date = df["fecha_alta"].dropna().median()
        df.loc[still_missing, "fecha_alta"] = median_date
        print(f"Vẫn dùng median date cho {still_missing.sum()} dòng thiếu cả antiguedad")

### 4.6 Xử lý `indrel`
Fill giá trị thiếu bằng trạng thái phổ biến nhất (mode).

In [ ]:
print(df["indrel"].value_counts(dropna=False))

most_common = df["indrel"].mode(dropna=True)
fill_value = most_common.iloc[0] if len(most_common) else 1
df.loc[df["indrel"].isnull(), "indrel"] = fill_value

### 4.7 Loại bỏ cột không cần thiết
`tipodom` không hữu ích, `cod_prov` dư thừa vì đã có tên tỉnh ở `nomprov`, `conyuemp` vì quá nhiều null.

In [ ]:
df = df.drop(columns=["tipodom", "cod_prov", "conyuemp"], errors="ignore")
df.isnull().sum()[df.isnull().sum() > 0]

### 4.8 Xử lý `ind_actividad_cliente`
Fill bằng 1.

In [ ]:
df.loc[df["ind_actividad_cliente"].isnull(), "ind_actividad_cliente"] = 1

### 4.9 Xử lý `nomprov` (tên tỉnh)

**Fix**: bản gốc dùng chuỗi byte kiểu Python 2 (`"CORU\xc3\x91A, A"`) để sửa lỗi
encode của "CORUÑA" — cách này không match được trong Python 3. Sửa bằng ký tự
unicode `\u00d1` trực tiếp.

In [ ]:
df["nomprov"] = df["nomprov"].replace({"CORU\u00d1A, A": "CORUNA, A"})
df.loc[df["nomprov"].isnull(), "nomprov"] = "UNKNOWN"
df["nomprov"].unique()

### 4.10 Xử lý `renta` (thu nhập)

Thu nhập biến động nhiều theo tỉnh, nên impute theo median của từng tỉnh sẽ chính
xác hơn median toàn cục. Trực quan hoá trước:

In [ ]:
income_by_province = (
    df.loc[df["renta"].notnull()].groupby("nomprov")["renta"].median().sort_values()
)

plt.figure(figsize=(12, 6))
sns.barplot(x=income_by_province.index, y=income_by_province.values, color="#c60b1e")
plt.xticks(rotation=90)
plt.ylabel("Median Income")
plt.xlabel("Province")
plt.title("Income Distribution by Province")
plt.tight_layout()
plt.show()

**Fix bug**: bản gốc impute bằng cách `merge()` rồi gán qua `.reset_index()`,
làm lệch index giữa hai DataFrame nên gán nhầm giá trị cho nhiều hàng. Sửa bằng
`groupby().transform("median")` — đảm bảo alignment luôn đúng theo index gốc.

In [ ]:
province_median = df.groupby("nomprov")["renta"].transform("median")
df["renta"] = df["renta"].fillna(province_median)

# fallback nếu cả tỉnh không có giá trị nào để tính median
df["renta"] = df["renta"].fillna(df["renta"].median())

### 4.11 Xử lý `ind_nomina_ult1` / `ind_nom_pens_ult1`

Vì đây là time-series theo từng khách hàng theo
tháng, forward-fill theo lịch sử `ncodpers`

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])
for col in ["ind_nomina_ult1", "ind_nom_pens_ult1"]:
    df[col] = df.groupby("ncodpers")[col].transform(lambda s: s.ffill())
    df[col] = df[col].fillna(0)

### 4.12 Các cột dạng chuỗi còn thiếu
Điền "UNKNOWN" hoặc giá trị hợp lý nhất tuỳ theo ý nghĩa từng cột.

In [ ]:
string_cols = df.select_dtypes(include=["object", "string"]).columns
missing_cols = [c for c in string_cols if df[c].isnull().any()]
print("Các cột còn thiếu:", missing_cols)

if "indfall" in missing_cols:
    df.loc[df["indfall"].isnull(), "indfall"] = "N"
if "tiprel_1mes" in missing_cols:
    df.loc[df["tiprel_1mes"].isnull(), "tiprel_1mes"] = "A"
    df["tiprel_1mes"] = df["tiprel_1mes"].astype("category")

if "indrel_1mes" in df.columns:
    map_dict = {
        1.0: "1", "1.0": "1", "1": "1",
        3.0: "3", "3.0": "3", "3": "3",
        2.0: "2", "2.0": "2", "2": "2",
        4.0: "4", "4.0": "4", "4": "4",
        "P": "P",
    }
    df["indrel_1mes"] = df["indrel_1mes"].fillna("P")
    df["indrel_1mes"] = df["indrel_1mes"].map(lambda x: map_dict.get(x, x))
    df["indrel_1mes"] = df["indrel_1mes"].astype("category")

remaining = [c for c in missing_cols if c not in ("indfall", "tiprel_1mes", "indrel_1mes")]
for col in remaining:
    df.loc[df[col].isnull(), col] = "UNKNOWN"

df.isnull().sum()[df.isnull().sum() > 0]

### 4.13 Kiểm tra lại — xác nhận hết missing

In [ ]:
remaining_na = df.isnull().sum()[df.isnull().sum() > 0]
remaining_na if len(remaining_na) else "Không còn cột nào thiếu dữ liệu."

### 4.14 Convert các cột sản phẩm (`ind_*_ult1`) sang kiểu int

In [ ]:
feature_cols = df.filter(regex="ind_.*ult.*").columns
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

feature_cols

## Checkpoint — lưu cho các notebook tiếp theo
Lưu `df` đã làm sạch để `03_eda_after_cleaning.ipynb` và `04_feature_engineering.ipynb`
đọc tiếp — cả 2 notebook đó đều nhánh ra từ đúng bản dữ liệu sạch này.

In [ ]:
df.to_parquet("data/interim/cleaned.parquet", index=False)
print(f"Đã lưu checkpoint: {df.shape}")

---
# ============================================================
# PHẦN 3: EDA SAU KHI LÀM SẠCH (POST-CLEANING EDA)
# Nguồn: `03_eda_after_cleaning.ipynb`
# Mô tả: Phân tích chuyên sâu 24 sản phẩm ngân hàng, phân phối theo demographic và rút ra 7 key findings.
# ============================================================


# 3. EDA — sau khi làm sạch
Tiếp nối từ `02_cleaning.ipynb` — đọc lại checkpoint `cleaned.parquet`.
Notebook này chỉ phân tích, không tạo feature mới (feature engineering nằm ở
`04_feature_engineering.ipynb`, nhánh riêng từ cùng checkpoint này).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/cleaned.parquet")
print(f"Loaded checkpoint: {df.shape}")

## 5. EDA — sau khi làm sạch
Hai phân tích này cần chạy sau bước làm sạch (Mục 4) vì phụ thuộc `feature_cols` đã là 0/1 kiểu int sạch — đặt riêng thành mục 5 thay vì trộn vào Feature Engineering, vì bản chất đây vẫn là phân tích/EDA, không tạo ra feature mới.

### 5.1 Tương quan giữa 24 sản phẩm
Tính trên **snapshot tháng gần nhất của mỗi khách hàng** (không phải toàn bộ panel) để tránh 1 khách hàng đóng góp nhiều dòng lặp lại qua các tháng, gây lệch hệ số tương quan.

In [ ]:
# Snapshot gần nhất mỗi khách hàng, tránh double-count theo tháng
df_last_snapshot = df.loc[df.groupby("ncodpers")["fecha_dato"].idxmax()]

corr_products = df_last_snapshot[feature_cols].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_products, cmap="coolwarm", center=0, square=True,
            xticklabels=True, yticklabels=True)
plt.title("Tương quan giữa 24 sản phẩm (snapshot tháng gần nhất mỗi khách hàng)")
plt.tight_layout()
plt.show()

# Top các cặp sản phẩm tương quan dương mạnh nhất (bỏ đường chéo)
import numpy as np
corr_pairs = (
    corr_products.where(np.triu(np.ones(corr_products.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print("Top 10 cặp sản phẩm tương quan dương mạnh nhất:")
print(corr_pairs.head(10))

print("Top 10 cặp sản phẩm tương quan âm mạnh nhất:")
print(corr_pairs.sort_values(ascending=True).head(10))


### 5.2 Tỷ lệ sở hữu từng sản phẩm (class balance)

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]

ownership_rate = (
    df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).mean().sort_values(ascending=False) * 100
)

plt.figure(figsize=(10, 8))
sns.barplot(x=ownership_rate.values, y=ownership_rate.index, color="mediumpurple")
plt.xlabel("% khách hàng sở hữu")
plt.title("Tỷ lệ sở hữu từng sản phẩm (class balance của từng target)")
plt.tight_layout()
plt.show()

ownership_rate

"24 sản phẩm có ownership rate từ 0% đến 67.3%, cực kỳ mất cân bằng. 3 sản phẩm (ind_aval, ind_cder, ind_ahor) gần như không xuất hiện trong sample — cần chạy trên full dataset hoặc loại khỏi tập target ở Checkpoint 2. scale_pos_weight đã tính sẵn để dùng trực tiếp cho XGBoost/LightGBM."

### 5.3 Số sản phẩm sở hữu theo nhóm tuổi (age)
Chia `age` thành nhóm để dễ nhìn xu hướng thay vì biểu đồ scatter rối mắt.
`n_products` tính lại ở đây (đã bị xoá ở mục 3.9 vì đó là EDA trên dữ liệu thô) —
giữ lại xuyên suốt 5.3-5.7, xoá ở cuối 5.7.

In [ ]:
product_cols = [c for c in df.columns if c.startswith("ind_") and c.endswith("_ult1")]
df["n_products"] = df[product_cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)

n_products_by_age = df.groupby("age_group", observed=True)["n_products"].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=n_products_by_age.index, y=n_products_by_age.values, color="steelblue")
plt.title("Số sản phẩm trung bình theo nhóm tuổi")
plt.xlabel("Nhóm tuổi")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_age

**Finding**: Quan hệ không tuyến tính theo tuổi — số sản phẩm TB tăng dần từ
nhóm <25 (**0.92**) lên đỉnh ở nhóm 45-54 tuổi (**2.02**), rồi giảm nhẹ ở 55-64 (1.87)
và 65+ (1.60). Không phải "càng lớn tuổi càng nhiều sản phẩm" — đỉnh nằm ở độ tuổi trung
niên, khớp với giai đoạn nhu cầu tài chính (vay mua nhà, tiết kiệm hưu trí...) cao nhất.

### 5.4 Số sản phẩm sở hữu theo thu nhập (renta)
Chia theo tứ phân vị (`qcut`) thay vì khoảng đều (`cut`) vì `renta` lệch phải rất mạnh
(một số ít khách hàng thu nhập rất cao) — chia đều khoảng sẽ dồn gần hết dữ liệu vào 1 bin.

In [ ]:
df["renta_group"] = pd.qcut(df["renta"], q=4, labels=["Q1 (thấp nhất)", "Q2", "Q3", "Q4 (cao nhất)"])

n_products_by_renta = df.groupby("renta_group", observed=True)["n_products"].mean()

plt.figure(figsize=(7, 4))
sns.barplot(x=n_products_by_renta.index, y=n_products_by_renta.values, color="darkorange")
plt.title("Số sản phẩm trung bình theo tứ phân vị thu nhập")
plt.xlabel("Nhóm thu nhập (renta)")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_renta

**Finding**: Thu nhập tương quan dương với số sản phẩm nhưng không mạnh bằng
thâm niên — tăng đều từ **1.35** (Q1 thấp nhất) lên **1.80** (Q4 cao nhất). Chênh lệch
~33% giữa nhóm cao nhất và thấp nhất, so với ~119% ở biến thâm niên (5.5).

### 5.5 Số sản phẩm sở hữu theo thâm niên (antiguedad)
`antiguedad` tính theo tháng. Chia nhóm theo mốc thường dùng trong phân tích khách hàng
ngân hàng: khách mới (<1 năm), 1-3 năm, 3-5 năm, 5-10 năm, 10 năm+.

In [ ]:
seniority_bins = [-1, 12, 36, 60, 120, 1000]
seniority_labels = ["<1 năm", "1-3 năm", "3-5 năm", "5-10 năm", "10 năm+"]
df["antiguedad_group"] = pd.cut(df["antiguedad"], bins=seniority_bins, labels=seniority_labels)

n_products_by_seniority = df.groupby("antiguedad_group", observed=True)["n_products"].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=n_products_by_seniority.index, y=n_products_by_seniority.values, color="seagreen")
plt.title("Số sản phẩm trung bình theo thâm niên khách hàng")
plt.xlabel("Thâm niên")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

n_products_by_seniority

**Finding**: Đây là biến ảnh hưởng mạnh nhất trong 3 biến demographic đã xét —
khách hàng thâm niên 10 năm+ sở hữu TB **2.21 sản phẩm**, gấp hơn 2 lần khách mới <1 năm
(**1.01 sản phẩm**). Quan hệ tăng đều đặn qua từng mốc (1.01 → 1.14 → 1.21 → 1.57 → 2.21),
gần như tuyến tính theo độ gắn bó.

### 5.6 Số sản phẩm sở hữu theo kênh đăng ký (canal_entrada)
`canal_entrada` cardinality cao (hàng chục kênh) — chỉ xem top 10 kênh phổ biến nhất
để biểu đồ còn đọc được, thay vì vẽ hết.

In [ ]:
top_channels = df["canal_entrada"].value_counts().head(10).index

n_products_by_channel = (
    df[df["canal_entrada"].isin(top_channels)]
    .groupby("canal_entrada", observed=True)["n_products"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
sns.barplot(x=n_products_by_channel.values, y=n_products_by_channel.index, color="mediumvioletred")
plt.title("Số sản phẩm trung bình theo top 10 kênh đăng ký")
plt.xlabel("Số sản phẩm trung bình")
plt.ylabel("canal_entrada")
plt.tight_layout()
plt.show()

n_products_by_channel

**Finding**: Cross-sell chênh lệch rất mạnh giữa các kênh đăng ký — kênh phổ biến
nhất (~30% khách hàng) lại có số sản phẩm TB **thấp nhất**, trong khi 1 kênh nhỏ hơn
(~25% khách hàng) có số sản phẩm TB **cao gấp hơn 2 lần**. Gợi ý chất lượng/mục đích
khách hàng đến từ mỗi kênh rất khác nhau, đáng làm feature quan trọng cho model.

*(Số liệu thật ở đây tính từ bản đã frequency-encode vì mình chỉ có file parquet đã xử
lý xong, không có bản `canal_entrada` thô — khi bạn tự chạy cell này trên `df` thật sẽ
ra đúng tên kênh cụ thể thay vì giá trị tần suất, nhưng pattern chênh lệch giữa các kênh
nên vẫn giữ nguyên.)*

### 5.7 Tương quan giữa đặc điểm khách hàng (input) và từng sản phẩm (output)
Mở rộng từ 5.1 (chỉ tương quan **giữa 24 sản phẩm với nhau**) và 5.3-5.6 (chỉ xem input vs `n_products` gộp chung) sang tương quan giữa **từng input demographic** và **từng sản phẩm riêng lẻ** — giúp biết chính xác đặc điểm nào ảnh hưởng đến sản phẩm cụ thể nào, thay vì chỉ biết ảnh hưởng đến tổng số sản phẩm.

Dùng point-biserial correlation (tương đương Pearson khi 1 biến là binary 0/1) — tính trên `df_last_snapshot` (đã có từ 5.1) để tránh double-count theo tháng.

In [ ]:
# Input: demographic/behavioral features. Output: 24 sản phẩm (product_cols)
numeric_inputs = df_last_snapshot[["age", "renta", "antiguedad", "ind_actividad_cliente", "indrel"]].copy()

# Encode tạm categorical ít category CHỈ để tính correlation ở đây — không ghi đè df gốc
# (encode chính thức cho model nằm ở 04_feature_engineering, notebook này chỉ phân tích)
sexo_dummy = pd.get_dummies(df_last_snapshot["sexo"], prefix="sexo")
segmento_dummy = pd.get_dummies(df_last_snapshot["segmento"], prefix="segmento")

input_features = pd.concat([numeric_inputs, sexo_dummy, segmento_dummy], axis=1)
output_features = df_last_snapshot[product_cols].apply(pd.to_numeric, errors="coerce")

combined = pd.concat([input_features, output_features], axis=1)
full_corr = combined.corr()
input_output_corr = full_corr.loc[input_features.columns, output_features.columns]

plt.figure(figsize=(16, 8))
sns.heatmap(input_output_corr, cmap="RdBu_r", center=0, xticklabels=True, yticklabels=True,
            cbar_kws={"label": "Correlation"})
plt.title("Tương quan input (đặc điểm khách hàng) x output (sản phẩm)")
plt.xlabel("Sản phẩm (output)")
plt.ylabel("Đặc điểm khách hàng (input)")
plt.tight_layout()
plt.show()

In [ ]:
# Bảng thống kê: top 15 cặp (input feature, sản phẩm) tương quan mạnh nhất (trị tuyệt đối)
corr_pairs = input_output_corr.unstack().sort_values(key=abs, ascending=False)
corr_table = corr_pairs.head(15).rename("correlation").reset_index()
corr_table.columns = ["input_feature", "product", "correlation"]
corr_table

**Finding**: *(điền sau khi chạy trên data thật — đọc bảng `corr_table` ở trên để lấy đúng số. Mẫu câu: "Đặc điểm **X** tương quan mạnh nhất với sản phẩm **Y** (hệ số **Z**), trong khi các sản phẩm còn lại gần như không liên hệ với input demographic (hệ số <0.1) — gợi ý model nên dựa nhiều vào lag features (hành vi quá khứ) hơn là demographic tĩnh cho phần lớn sản phẩm.")*

### 5.8 Xu hướng số sản phẩm theo thời gian (fecha_dato)
Xem số sản phẩm trung bình/khách hàng có đổi theo tháng không (mùa vụ, xu hướng tăng
trưởng chung...). Dọn các cột tạm (`age_group`, `renta_group`, `antiguedad_group`,
`n_products`) ở cuối vì chỉ dùng cho EDA, không phải feature cho model.

In [ ]:
n_products_by_month = df.groupby("fecha_dato")["n_products"].mean()

plt.figure(figsize=(10, 4))
n_products_by_month.plot(marker="o", color="teal")
plt.title("Số sản phẩm trung bình / khách hàng theo tháng")
plt.xlabel("Tháng")
plt.ylabel("Số sản phẩm trung bình")
plt.tight_layout()
plt.show()

df.drop(columns=["age_group", "renta_group", "antiguedad_group", "n_products"], inplace=True)
n_products_by_month

**Finding**: Số sản phẩm TB giảm dần rõ rệt theo thời gian — ổn định quanh 1.78-1.80
từ 01/2015-06/2015, tụt xuống 1.35-1.40 từ 07/2015-01/2016, rồi giảm mạnh còn 0.91 vào
02/2016 (tháng cuối trong dữ liệu).

**Lưu ý quan trọng**: pattern này CÓ THỂ là artifact của cách sample dữ liệu
(`LIMIT_ROWS=10_000_000` chỉ đọc phần đầu file gốc theo thứ tự thời gian,
`LIMIT_PEOPLE=10_000` sample khách hàng 1 lần duy nhất) chứ chưa chắc phản ánh đúng
hành vi mùa vụ thật của khách hàng — cần verify lại trên dữ liệu đầy đủ (bỏ `LIMIT_ROWS`)
trước khi dùng để thiết kế feature theo mùa vụ ở checkpoint 2/3.

## 5.9 Tổng hợp finding chính

Gộp lại 7 finding quan trọng nhất từ toàn bộ EDA (3.x + 5.x) để dùng cho báo cáo/trình bày:

1. **Ownership cực kỳ lệch giữa 24 sản phẩm** — `ind_cco_fin_ult1` (tài khoản vãng lai)
   chiếm ~67% số dòng, trong khi 4 sản phẩm hiếm nhất (`ind_ahor`, `ind_aval`, `ind_deco`,
   `ind_deme`) gần như 0%. Nên loại 4 sản phẩm này khi train model (khớp với cách các
   solution top của competition gốc xử lý).

2. **Cụm sản phẩm tương quan gần như tuyệt đối** — `ind_nomina_ult1` (lương) và
   `ind_nom_pens_ult1` (lương hưu) có hệ số tương quan **0.977**, gần như luôn đi cùng
   nhau. Cần lưu ý tránh dư thừa feature/target khi thiết kế model.

3. **Thâm niên là biến demographic ảnh hưởng mạnh nhất** đến số sản phẩm sở hữu — khách
   10 năm+ sở hữu TB **2.21** sản phẩm, gấp hơn 2 lần khách mới <1 năm (**1.01**), tăng
   gần như tuyến tính qua từng mốc thời gian gắn bó.

4. **Tuổi có quan hệ không tuyến tính** với số sản phẩm — đỉnh ở nhóm 45-54 tuổi
   (**2.02** sản phẩm TB), không phải nhóm cao tuổi nhất (65+ chỉ **1.60**).

5. **Thu nhập (renta) tương quan dương nhưng yếu hơn thâm niên** — chênh lệch ~33%
   giữa Q4 (1.80) và Q1 (1.35), so với ~119% ở biến thâm niên.

6. **Kênh đăng ký (canal_entrada) tạo chênh lệch cross-sell rất lớn** — kênh phổ biến
   nhất có số sản phẩm TB thấp nhất, một kênh khác nhỏ hơn lại cao gấp hơn 2 lần. Đáng
   làm feature quan trọng.

7. **Target cực kỳ mất cân bằng** — chỉ ~0.68% các cặp khách hàng-sản phẩm-tháng là
   "Added" (mua mới). Đây là đặc tính bản chất của bài toán, không phải lỗi dữ liệu —
   cần xử lý bằng weighting/undersampling ở bước train (checkpoint 2), không phải
   accuracy mà phải dùng MAP@7/AUC để đánh giá.

**Lưu ý riêng cho finding #6-#7 ở mục 5.8**: pattern giảm dần theo thời gian có thể là
artifact của việc sample dữ liệu (`LIMIT_ROWS`, `LIMIT_PEOPLE`) — cần verify lại trên
dữ liệu đầy đủ trước khi dùng để thiết kế feature mùa vụ.

---
# ============================================================
# PHẦN 4: FEATURE ENGINEERING & CHUẨN BỊ DỮ LIỆU MODELING
# Nguồn: `04_feature_engineering.ipynb`
# Mô tả: Tạo lag features, mã hóa categorical, xác định nhãn Added/Dropped/Maintained và melt sang long format.
# ============================================================


# 4. Feature Engineering
Tiếp nối từ `02_cleaning.ipynb` — đọc lại checkpoint `cleaned.parquet` (nhánh riêng
từ `03_eda_after_cleaning.ipynb`, không phụ thuộc notebook đó).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

In [ ]:
df = pd.read_parquet("data/interim/cleaned.parquet")
print(f"Loaded checkpoint: {df.shape}")

In [ ]:
feature_cols = df.filter(regex="ind_.*ult.*").columns
feature_cols

## 6. Feature Engineering
Gộp toàn bộ các bước tạo feature mới vào một mạch: lag sản phẩm → lag/cờ thay đổi cho các cột hành vi → nhãn Added/Dropped/Maintained → long format → rút gọn lag dư thừa.

### 6.1 Lag features cho sản phẩm (lag 1–5)
Với mỗi sản phẩm, tạo 5 cột lag (trạng thái 1–5 tháng trước) để dùng làm feature và để tính Added/Dropped/Maintained ở bước sau.

In [ ]:
assert df[feature_cols].isin([0, 1]).all().all(), (
    "feature_cols không còn là 0/1 nhị phân — có thể đã chạy nhầm cell "
    "Added/Dropped trước cell lag này, hoặc kernel còn state cũ. "
    "Restart kernel và Run All lại từ đầu."
)

N_LAGS = 5
lag_dict = {}
for col in feature_cols:
    for lag in range(1, N_LAGS + 1):
        lag_dict[f"{col}_lag_{lag}"] = df.groupby("ncodpers")[col].shift(lag).fillna(0).astype(int)

lag_df = pd.DataFrame(lag_dict)
df = pd.concat([df, lag_df], axis=1)

### 6.2 Lag + cờ "changed" cho `segmento` / `ind_actividad_cliente` / `tiprel_1mes`

Các cột này phản ánh trạng thái/quan hệ của khách hàng theo tháng — bản thân sự **thay đổi** (vừa active lại, vừa đổi segment, đổi loại quan hệ...) là tín hiệu cross-sell mạnh hơn giá trị hiện tại (ví dụ khách vừa active lại có khả năng cao sắp phát sinh giao dịch/mua sản phẩm mới). Không cần đủ 5 lag như sản phẩm, chỉ cần `lag_1` + cờ nhị phân `changed` là đủ để bắt tín hiệu này.

In [ ]:
df = df.sort_values(["ncodpers", "fecha_dato"])

behavior_cols = [c for c in ["segmento", "ind_actividad_cliente", "tiprel_1mes"] if c in df.columns]
for col in behavior_cols:
    lag_col = f"{col}_lag_1"
    df[lag_col] = df.groupby("ncodpers")[col].shift(1)
    df[f"{col}_changed"] = (df[col] != df[lag_col]).astype(int)
    # Tháng đầu tiên của mỗi khách hàng không có lag -> không tính là "changed"
    df.loc[df[lag_col].isnull(), f"{col}_changed"] = 0

df[[f"{c}_changed" for c in behavior_cols]].mean()

### 6.2b — Encode categorical 

In [ ]:
# One-hot cho các cột nominal cardinality thấp
# Fix: thêm segmento_lag_1 / tiprel_1mes_lag_1 (bản tháng trước, tạo ở 6.2)
# vào cùng danh sách encode -- trước đó bị sót, vẫn còn ở dạng string/NaN thô
# nên không đưa thẳng vào model được.
low_card_encode = [
    "segmento", "sexo", "tiprel_1mes", "indrel_1mes",
    "segmento_lag_1", "tiprel_1mes_lag_1",
]
low_card_encode = [c for c in low_card_encode if c in df.columns]

# NaN ở *_lag_1 (khách chưa có tháng trước đó, vd. tháng đầu xuất hiện) ->
# coi là "UNKNOWN", nhất quán với cách xử lý missing của bản hiện tại
# (đã điền "UNKNOWN" ở bước 4.12).
for col in low_card_encode:
    if df[col].isnull().any():
        df[col] = df[col].astype(object).fillna("UNKNOWN")

cols_before_onehot = set(df.columns)
df = pd.get_dummies(df, columns=low_card_encode, prefix=low_card_encode, dtype="int8")
# Lấy đúng tên cột dummy mới sinh ra bằng cách diff trước/sau df.columns,
# KHÔNG dùng c.startswith(prefix) để gom cột: "segmento" và "tiprel_1mes"
# cũng là tiền tố của các cột lag/changed đã tạo ở bước 6.2
# (segmento_lag_1, tiprel_1mes_changed, ...) -> lọc theo prefix sẽ vô tình
# gom nhầm các cột đó vào nhóm one-hot.
onehot_cols = sorted(set(df.columns) - cols_before_onehot)

# Frequency encoding cho cột nominal cardinality cao
high_card_cols = ["canal_entrada", "pais_residencia", "nomprov"]
high_card_cols = [c for c in high_card_cols if c in df.columns]

freq_cols = []
for col in high_card_cols:
    freq = df[col].value_counts(normalize=True)
    df[f"{col}_freq"] = df[col].map(freq).astype("float32")
    freq_cols.append(f"{col}_freq")

# Bỏ cột string gốc sau khi đã có bản freq-encoded, tránh còn sót string
# cardinality cao trong output cuối (df_long / parquet) mà model không
# dùng trực tiếp được -- nếu muốn giữ lại để dễ debug/đọc, có thể bỏ dòng
# drop này, feature_columns ở cell dưới vẫn chỉ chọn đúng cột *_freq.
df = df.drop(columns=high_card_cols)

In [ ]:
# Cột binary S/N -> map thẳng 0/1
# Lưu ý: "indfall" KHÔNG còn nằm trong binary_cols nữa. Ở bước 4.1 (lọc
# khách hàng đã mất) toàn bộ dòng indfall == "S" đã bị loại khỏi df, nên
# tại thời điểm này cột indfall chỉ còn duy nhất giá trị "N" -- encode ra
# sẽ là một cột hằng số toàn 0, không còn giá trị dự đoán. Bỏ hẳn cột này
# thay vì giữ lại một feature "chết".
binary_cols = ["indresi", "indext"]
binary_cols = [c for c in binary_cols if c in df.columns]
for col in binary_cols:
    df[col] = df[col].map({"S": 1, "N": 0}).fillna(0).astype("int8")

if "indfall" in df.columns:
    df = df.drop(columns=["indfall"])

# ind_empleado: cardinality thấp (~5 giá trị: A/B/F/N/S) -> one-hot
cols_before_ind_empleado = set(df.columns)
if "ind_empleado" in df.columns:
    df = pd.get_dummies(df, columns=["ind_empleado"], prefix="ind_empleado", dtype="int8")
ind_empleado_cols = sorted(set(df.columns) - cols_before_ind_empleado)

# ult_fec_cli_1t: gần như toàn NaN/"UNKNOWN" -> bản thân giá trị ngày không
# có ý nghĩa dự đoán nhiều, chỉ cần biết CÓ hay KHÔNG có giá trị (khách từng
# đổi trạng thái primary customer) là đủ tín hiệu
if "ult_fec_cli_1t" in df.columns:
    df["has_ult_fec_cli_1t"] = (~df["ult_fec_cli_1t"].isin([np.nan, "UNKNOWN"])).astype("int8")
    df = df.drop(columns=["ult_fec_cli_1t"])

In [ ]:
# Tổng hợp feature_columns -- dùng chính các biến vừa tạo ở 2 cell trên
# (onehot_cols, freq_cols, ind_empleado_cols, binary_cols), KHÔNG còn tham
# chiếu tới train_df / lag1_cols / demo_cols / cat_encoded_cols "từ trên
# trời rơi xuống" như bản gốc -- đó chính là nguyên nhân NameError khi chạy
# (những biến này chưa từng được định nghĩa ở đâu trong notebook).

# lag_1 của 24 sản phẩm: tín hiệu portfolio hiện tại, đã tạo ở bước 6.1
lag1_cols = [f"{col}_lag_1" for col in feature_cols]

# Biến demographic / hành vi dạng số
demo_cols = [
    "age", "antiguedad", "renta", "ind_nuevo", "indrel",
    "ind_actividad_cliente", "month",
] + [f"{c}_changed" for c in behavior_cols]
demo_cols = [c for c in demo_cols if c in df.columns]

# Categorical đã encode ở 2 cell trên (one-hot + frequency)
cat_encoded_cols = onehot_cols + freq_cols + ind_empleado_cols

extra_cols = ["has_ult_fec_cli_1t"] + binary_cols
extra_cols = [c for c in extra_cols if c in df.columns]

feature_columns = lag1_cols + demo_cols + cat_encoded_cols + extra_cols
print(f"Tổng số feature: {len(feature_columns)}")
feature_columns

### 6.3 Added / Dropped / Maintained

Với mỗi khách hàng và mỗi sản phẩm, xác định trong tháng đó khách hàng đã thêm
mới, huỷ bỏ, hay giữ nguyên sản phẩm — bằng cách lấy diff giữa các tháng liên
tiếp. **Đảm bảo sort theo (`ncodpers`, `month_id`) trước khi diff** để tránh
trường hợp dữ liệu không theo đúng thứ tự thời gian (bản gốc giả định thứ tự đã
đúng sẵn).

In [ ]:
unique_months = df["fecha_dato"].drop_duplicates().sort_values().reset_index(drop=True)
month_id_map = {date: i + 1 for i, date in enumerate(unique_months)}
df["month_id"] = df["fecha_dato"].map(month_id_map)
df = df.sort_values(["ncodpers", "month_id"])

# Vectorized: dùng lag_1 đã tính ở bước lag feature, không cần groupby().transform() nữa
# Fix: tách "Maintained" thành 2 loại rõ ràng thay vì gộp chung, vì ý nghĩa khác hẳn nhau:
#   - Maintained_NotOwned (prev=0, current=0): chưa sở hữu, vẫn chưa sở hữu -> nhãn 0 quan trọng
#     nhất cho task "dự đoán sản phẩm MỚI", KHÔNG được lọc bỏ.
#   - Maintained_Owned (prev=1, current=1): đã sở hữu từ trước, vẫn giữ -> không thuộc phạm vi
#     task "thêm sản phẩm mới" (khách đã có sẵn rồi).
for col in feature_cols:
    lag_col = f"{col}_lag_1"
    current = df[col]
    prev = df[lag_col]  # đã fillna(0) sẵn từ bước tạo lag
    df[col] = np.select(
        [
            (current == 1) & (prev == 0),
            (current == 0) & (prev == 1),
            (current == 1) & (prev == 1),
        ],
        ["Added", "Dropped", "Maintained_Owned"],
        default="Maintained_NotOwned",
    )

### 6.4 Chuyển sang long format (melt)

**Fix (bug nghiêm trọng)**: bản gốc lọc bỏ hết các dòng "Maintained" ở bước này, chỉ giữ
lại Added/Dropped. Nhưng bài toán gốc là dự đoán sản phẩm **MỚI** sẽ được thêm — cần có
nhãn 0 (khách chưa sở hữu tháng trước, vẫn chưa sở hữu tháng này = `Maintained_NotOwned`)
thì model mới học được thế nào là "bình thường" trước khi phân biệt được trường hợp sắp
mua thêm. Lọc bỏ hết "Maintained" nghĩa là mất hết nhãn 0 đúng nghĩa, chỉ còn toàn các sự
kiện thay đổi (Added/Dropped) — model sẽ học sai bài toán.

**Cách dùng ở checkpoint 2** (train model "recommend sản phẩm mới"): lọc lấy các dòng có
`status` thuộc `{Added, Maintained_NotOwned}` (tức prev=0 — chưa sở hữu trước đó), rồi gán
`label = 1` nếu `Added`, `label = 0` nếu `Maintained_NotOwned`. Các dòng `Dropped` /
`Maintained_Owned` (đã sở hữu từ trước) nằm ngoài phạm vi task này, nhưng vẫn giữ lại ở
đây phòng khi cần cho phân tích churn.

Lưu ý: giữ toàn bộ sẽ làm file lớn hơn nhiều so với bản trước (chỉ ~21.8k dòng vì đã lọc
mất phần lớn dữ liệu) — với ~7.6k khách × ~14 tháng × 24 sản phẩm sẽ ra khoảng vài triệu
dòng. Nếu quá nặng cho máy, có thể downsample `Maintained_NotOwned` sau (giữ toàn bộ
Added/Dropped + sample ngẫu nhiên một phần Maintained_NotOwned), nhưng nên làm ở bước
train (checkpoint 2), không nên lọc mất tại bước lưu dữ liệu gốc này.

In [ ]:
df_long = df.melt(
    id_vars=[c for c in df.columns if c not in feature_cols],
    value_vars=list(feature_cols),
    var_name="product",
    value_name="status",
)

# Fix: KHÔNG lọc bỏ "Maintained_*" nữa — giữ toàn bộ để có đủ nhãn 0 cho training.
print(df_long["status"].value_counts())
df_long.shape

### 6.5 Rút gọn lag dư thừa

Sau melt, mỗi dòng chỉ nói về **1 sản phẩm** (cột `product`) nhưng vẫn mang theo `lag_2`–`lag_5` của cả 24 sản phẩm khác — dư thừa. Giữ nguyên `lag_1` của toàn bộ 24 sản phẩm (tín hiệu portfolio hiện tại, quan trọng cho cross-sell), chỉ rút gọn `lag_2`–`lag_5` về đúng 4 cột `self_lag_2`…`self_lag_5` ứng với sản phẩm đang xét ở từng dòng.

In [ ]:
product_list = list(feature_cols)
col_index = {p: i for i, p in enumerate(product_list)}
prod_idx = df_long["product"].map(col_index).to_numpy()
row_idx = np.arange(len(df_long))

for lag in range(2, N_LAGS + 1):
    lag_cols = [f"{p}_lag_{lag}" for p in product_list]
    lag_matrix = df_long[lag_cols].to_numpy()
    df_long[f"self_lag_{lag}"] = lag_matrix[row_idx, prod_idx]
    df_long = df_long.drop(columns=lag_cols)

print(f"Shape sau khi rút gọn lag: {df_long.shape}")
df_long.filter(regex="lag").columns.tolist()

## 7. Lưu kết quả

In [ ]:
LOCAL_PATH = "cleaned_long_format.parquet"
df_long.to_parquet(LOCAL_PATH, engine="pyarrow", index=False)

df_long.head()